# 02. YOLO11: обучение и тест детектора

Детектор обучается только на полных изображениях Gunduz. Долгое обучение и однократный test включаются раздельно.

In [ ]:
from pathlib import Path
import os, subprocess, sys, yaml

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), None)
assert PROJECT_ROOT is not None
os.chdir(PROJECT_ROOT)
CONFIG = PROJECT_ROOT / 'configs/detector.yaml'
WEIGHTS = PROJECT_ROOT / 'artifacts/detector/yolo11m/weights/best.pt'
RUN_TRAINING = False
RUN_FINAL_TEST = False
ENABLE_CLEARML = False

## Effective configuration и данные

In [ ]:
from core.config_loader import load_config
config = load_config(CONFIG)
data_yaml = PROJECT_ROOT / config['paths']['data_yaml']
print(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
assert data_yaml.is_file(), f'Сначала выполните 01_prepare_data.ipynb: {data_yaml}'
data_config = yaml.safe_load(data_yaml.read_text(encoding='utf-8'))
for split in ('train', 'val', 'test'):
    split_file = Path(data_config['path']) / data_config[split]
    assert split_file.is_file() and split_file.read_text(encoding='utf-8').strip(), split_file
    print(split, 'images:', len(split_file.read_text(encoding='utf-8').splitlines()))
print('YOLO split preflight: OK')

## Обучение

In [ ]:
train_command = [sys.executable, '-m', 'scripts.run_train_detector', '--config', str(CONFIG)]
if ENABLE_CLEARML:
    train_command += ['--set', 'clearml.enabled=true']
if RUN_TRAINING:
    assert not WEIGHTS.exists(), f'Checkpoint уже существует: {WEIGHTS}'
    subprocess.run(train_command, check=True)
else:
    print('Training skipped:', ' '.join(map(str, train_command)))

## Однократный test
Не используйте test для подбора гиперпараметров.

In [ ]:
test_command = [sys.executable, '-m', 'scripts.run_test_detector', '--config', str(CONFIG), '--weights', str(WEIGHTS)]
if ENABLE_CLEARML:
    test_command += ['--set', 'clearml.enabled=true']
if RUN_FINAL_TEST:
    assert WEIGHTS.is_file(), WEIGHTS
    subprocess.run(test_command, check=True)
else:
    print('Final test skipped:', ' '.join(map(str, test_command)))

## Артефакты

In [ ]:
output = PROJECT_ROOT / config['paths']['output_dir']
for path in sorted(output.rglob('*')) if output.exists() else []:
    if path.is_file(): print(path.relative_to(PROJECT_ROOT))